# Official direct-sampling ARI data

This notebook downloads and analyzes the published Zenodo data used by *Rare Event Analysis of Large Language Models*. It does not generate samples, load a language model, use a GPU, or run SMC/TPS/MBAR. The archive stays in Google Drive and is reused across Colab sessions.

In [ ]:
# 1. Mount Drive for persistent data and outputs.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the repository, or fast-forward an existing clean checkout.
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git'
REPO_ROOT = Path('/content/rea_of_llms_supplementary_code-main')
if (REPO_ROOT / '.git').is_dir():
    dirty = subprocess.run(
        ['git', '-C', str(REPO_ROOT), 'status', '--porcelain'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError('Existing checkout has local changes; use a fresh Colab runtime.')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
print('repository:', REPO_ROOT)

In [ ]:
# 3. Enter and validate the repository root.
import os
os.chdir(REPO_ROOT)
assert (REPO_ROOT / 'rea_llms' / 'official_direct_data.py').is_file()
assert (REPO_ROOT / 'requirements-colab.txt').is_file()
print('cwd:', Path.cwd())

In [ ]:
# 4. Install lightweight analysis dependencies. No model is downloaded.
%pip install -q -r requirements-colab.txt

In [ ]:
# 5. EDIT PARAMETERS HERE.
DATA_ROOT = Path('/content/drive/MyDrive/rea_reference_data')
DOWNLOAD_IF_MISSING = True
VERIFY_CHECKSUM = True
TAIL_THRESHOLDS = '6,8,10,12,15'
OUTPUT_ROOT = Path('/content/drive/MyDrive/rea_reference_results')

DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# 6. Verify/download once, extract only direct ARI, and run the analysis.
import sys
command = [
    sys.executable, '-m', 'rea_llms.official_direct_data',
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--thresholds', TAIL_THRESHOLDS,
    '--require-raw-maximum',
]
if DOWNLOAD_IF_MISSING:
    command.append('--download-if-missing')
if not VERIFY_CHECKSUM:
    command.append('--no-verify-checksum')
subprocess.run(command, check=True)

In [ ]:
# 7. Display the direct summary and empirical tail table.
import json
import pandas as pd
from IPython.display import display

summary = json.loads((OUTPUT_ROOT / 'reference_direct_sampling_summary.json').read_text())
direct = summary['direct_ari']
display(pd.DataFrame([{
    'N': direct['count'],
    'representation': direct['ari_representation'],
    'mean': direct['mean'],
    'std': direct['standard_deviation_population'],
    'min': direct['min'],
    'median': direct['quartiles']['q50'],
    'max': direct['max'],
}]))
display(pd.DataFrame(direct['tails']))

In [ ]:
# 8. Show the single saved figure and the comparability report.
from IPython.display import Image, Markdown
display(Image(filename=str(OUTPUT_ROOT / 'reference_direct_sampling_boxplot.png')))
display(Markdown((OUTPUT_ROOT / 'REFERENCE_DIRECT_SAMPLING.md').read_text()))